# 🔍 SQL Murder Mystery Investigation

## Project Overview

This project demonstrates my SQL problem-solving and data investigation skills by solving a  murder mystery using a relational SQLite database.

The investigation begins with a crime scene report and follows a trail of evidence across multiple database tables to identify the murderer and the person responsible for hiring them.

## Skills Demonstrated

- SQL data exploration
- Filtering using WHERE, LIKE, IN, AND, and OR
- Sorting and limiting query results
- SQL JOIN operations
- Working with relational databases
- Analytical and investigative problem-solving

## Tools Used

- SQL (SQLite)
- Python
- Pandas
- Jupyter Notebook

## 1. Exploring the Database

In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
import sqlite3
conn = sqlite3.connect("sql-murder-mystery.db")

query = ("select * FROM sqlite_master")
data = pd.read_sql(query,conn)
(data)

,type,name,tbl_name,rootpage,sql
0,table,crime_scene_report,crime_scene_report,2,"CREATE TABLE crime_scene_report (\n date integer,\n type text,\n description text,\n city text\n )"
1,table,drivers_license,drivers_license,3,"CREATE TABLE drivers_license (\n id integer PRIMARY KEY,\n age integer,\n height integer,\n eye_color text,\n hair_color text,\n gender text,\n plate_number text,\n car_make text,\n car_model text\n )"
2,table,person,person,4,"CREATE TABLE person (\n id integer PRIMARY KEY,\n name text,\n license_id integer,\n address_number integer,\n address_street_name text,\n ssn integer,\n FOREIGN KEY (license_id) REFERENCES drivers_license(id)\n )"
3,table,facebook_event_checkin,facebook_event_checkin,5,"CREATE TABLE facebook_event_checkin (\n person_id integer,\n event_id integer,\n event_name text,\n date integer,\n FOREIGN KEY (person_id) REFERENCES person(id)\n )"
4,table,interview,interview,6,"CREATE TABLE interview (\n person_id integer,\n transcript text,\n FOREIGN KEY (person_id) REFERENCES person(id)\n )"
5,table,get_fit_now_member,get_fit_now_member,7,"CREATE TABLE get_fit_now_member (\n id text PRIMARY KEY,\n person_id integer,\n name text,\n membership_start_date integer,\n membership_status text,\n FOREIGN KEY (person_id) REFERENCES person(id)\n )"
6,index,sqlite_autoindex_get_fit_now_member_1,get_fit_now_member,8,None
7,table,get_fit_now_check_in,get_fit_now_check_in,9,"CREATE TABLE get_fit_now_check_in (\n membership_id text,\n check_in_date integer,\n check_in_time integer,\n check_out_time integer,\n FOREIGN KEY (membership_id) REFERENCES get_fit_now_member(id)\n )"
8,table,income,income,10,"CREATE TABLE income (\n ssn integer PRIMARY KEY,\n annual_income integer\n )"
9,table,solution,solution,11,"CREATE TABLE solution (\n user integer,\n value text\n )"


In [2]:
query =('select * from crime_scene_report')
pd.read_sql(query,conn)

,date,type,description,city
0,20180115,robbery,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,murder,Life? Dont talk to me about life.,Albany
2,20180115,murder,"Mama, I killed a man, put a gun against his head...",Reno
3,20180215,murder,REDACTED REDACTED REDACTED,SQL City
4,20180215,murder,Someone killed the guard! He took an arrow to the knee!,SQL City
...,...,...,...,...
1223,20180430,bribery,\n,Garden Grove
1224,20180430,fraud,‘Why not?’ said the March Hare.\n,Houma
1225,20180430,assault,\n,Fontana
1226,20180501,assault,"be NO mistake about it: it was neither more nor less than a pig, and she\n",Trenton


## 2. Finding the Crime Scene Report
## Clue ---> 
  ### A murder occurred in SQL City around January 2018.

In [3]:
query =("SELECT * FROM crime_scene_report WHERE date like '%201801%' AND type ='murder' AND city ='SQL City'")
df = pd.read_sql(query,conn)
df

,date,type,description,city
0,20180115,murder,"Security footage shows that there were 2 witnesses. The first witness lives at the last house on ""Northwestern Dr"". The second witness, named Annabel, lives somewhere on ""Franklin Ave"".",SQL City


In [4]:
query = ("select * from person")
pd.read_sql(query,conn)

,id,name,license_id,address_number,address_street_name,ssn
0,10000,Christoper Peteuil,993845,624,Bankhall Ave,747714076
1,10007,Kourtney Calderwood,861794,2791,Gustavus Blvd,477972044
2,10010,Muoi Cary,385336,741,Northwestern Dr,828638512
3,10016,Era Moselle,431897,1987,Wood Glade St,614621061
4,10025,Trena Hornby,550890,276,Daws Hill Way,223877684
...,...,...,...,...,...,...
10006,99936,Luba Benser,274427,680,Carnage Blvd,685095054
10007,99941,Roxana Mckimley,975942,1613,Gate St,512136801
10008,99965,Cherie Zeimantz,287627,3661,The Water Ave,362877324
10009,99982,Allen Cruse,251350,3126,N Jean Dr,348734531


## 3. Identifying the Witnesses
  ### Witness Number 1

In [5]:

query=("Select * from person where name like '%Annabel%' And address_street_name ='Franklin Ave'")
pd.read_sql(query,conn)

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


### Witness Number 2


In [6]:
query=('''Select * From person Where address_street_name = 'Northwestern Dr'
order by address_number DESC limit 1;
''')
pd.read_sql(query,conn)

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


## 4. Analyzing Witness Interviews

In [7]:
query =('''Select * from interview
where person_id in('16371','14887' )''')
pd.read_sql(query,conn)

,person_id,transcript
0,14887,"I heard a gunshot and then saw a man run out. He had a ""Get Fit Now Gym"" bag. The membership number on the bag started with ""48Z"". Only gold members have those bags. The man got into a car with a plate that included ""H42W""."
1,16371,"I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th."


### Key Clues

Based on the witness interviews, I identified several important clues about the suspect, including gym membership information and a partial vehicle plate number. These clues were used to narrow down the possible suspects.

## 5. Narrowing Down the Suspects

The gym membership table identified potential suspects, but additional personal information was required. I joined the membership and person         tables using `person_id` to retrieve the related license information and further narrow down the suspects.

In [8]:
query=(''' select g.id, person_id,g.name,license_id,membership_status	
from get_fit_now_member as g
join  person AS p
on p.id= g.person_id 
where g.id like '%48Z%' ''')
pd.read_sql(query,conn)

,id,person_id,name,license_id,membership_status
0,48Z38,49550,Tomas Baisley,309485,silver
1,48Z7A,28819,Joe Germuska,173289,gold
2,48Z55,67318,Jeremy Bowers,423327,gold


In [9]:
query=('''select * from drivers_license
where id in('309485' , '423327', '173289') 
And  plate_number like'%H42W%' ''')
pd.read_sql(query,conn)

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS


## 6. Identification of the murderer

In [10]:
query =('''Select * From person
where license_id in('423327') 

''')
pd.read_sql(query,conn)

,id,name,license_id,address_number,address_street_name,ssn
0,67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279


## 7. Murderer Interview

In [11]:
query =('''Select * From interview
where person_id in('67318') 

''')
pd.read_sql(query,conn)

,person_id,transcript
0,67318,"I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5"" (65"") or 5'7"" (67""). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.\n"


### Finding
#### The murderer admitted that he was hired by a woman to commit the murder.


## 8. Investigating the Person Behind the Crime

In [12]:
query =('''Select * From drivers_license
 where car_make ='Tesla' And gender ='female' and hair_color ='red'
''')
pd.read_sql(query,conn)

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,202298,68,66,green,red,female,500123,Tesla,Model S
1,291182,65,66,blue,red,female,08CM64,Tesla,Model S
2,918773,48,65,black,red,female,917UU3,Tesla,Model S


In [13]:
query =('''Select * From person
 where  license_id in("202298","291182","918773")
''')
pd.read_sql(query,conn)

,id,name,license_id,address_number,address_street_name,ssn
0,78881,Red Korb,918773,107,Camerata Dr,961388910
1,90700,Regina George,291182,332,Maple Ave,337169072
2,99716,Miranda Priestly,202298,1883,Golden Ave,987756388


### Clue
#### She attended the SQL Symphony Concert 3 times in December 2017

In [14]:
query =('''Select * From facebook_event_checkin 
where person_id in("78881","90700","99716") 
''')
pd.read_sql(query,conn)

,person_id,event_id,event_name,date
0,99716,1143,SQL Symphony Concert,20171206
1,99716,1143,SQL Symphony Concert,20171212
2,99716,1143,SQL Symphony Concert,20171229


## 9. Identification of the Mastermind 


In [15]:
query =('''Select * From person
where id in("99716") 
''')
pd.read_sql(query,conn)

,id,name,license_id,address_number,address_street_name,ssn
0,99716,Miranda Priestly,202298,1883,Golden Ave,987756388


## Final Conclusion

After analyzing the crime scene report, witness statements, gym membership records, vehicle information, and event attendance data, the investigation successfully identified the murderer as **Jeremy Bowers**.

Further investigation into the murderer's interview revealed additional clues about the person who hired them. By combining these clues with driver's license information and event records, **Miranda Priestly** was identified as the person responsible for  the crime.

This project demonstrated how SQL can be used to investigate relationships across multiple tables and progressively narrow down large datasets using evidence-based filtering.

### Key Takeaways

- Used SQL queries to follow an evidence-based investigation process.
- Connected related datasets using JOIN operations.
- Applied filtering techniques such as `WHERE`, `LIKE`, `IN`, `AND`, and `OR`.
- Translated unstructured clues into structured SQL queries.
- Used analytical reasoning to narrow multiple candidates to the final suspects.